In [1]:
def order_jobs_in_descending_order_of_total_completion_time(processing_times):
    total_completion_time = processing_times.sum(axis=1)
    return np.argsort(total_completion_time, axis=0).tolist()    

In [2]:
def insertion(sequence, position, value):
    new_seq = sequence[:]
    new_seq.insert(position, value)
    return new_seq

In [3]:
def neh_algorithm(processing_times):
    ordered_sequence = order_jobs_in_descending_order_of_total_completion_time(processing_times)
    # Define the initial order
    J1, J2 = ordered_sequence[:2]
    sequence = [J1, J2] if evaluate_sequence([J1, J2], processing_times) < evaluate_sequence([J2, J1], processing_times) else [J2, J1]
    del ordered_sequence[:2]
    # Add remaining jobs
    for job in ordered_sequence:
        Cmax = float('inf')
        best_sequence = []
        for i in range(len(sequence)+1):
            new_sequence = insertion(sequence, i, job)
            Cmax_eval = evaluate_sequence(new_sequence, processing_times)
            if Cmax_eval < Cmax:
                Cmax = Cmax_eval
                best_sequence = new_sequence
        sequence = best_sequence
    return sequence, Cmax

In [4]:
def evaluate_sequence(sequence, processing_times):
    _, num_machines = processing_times.shape
    num_jobs = len(sequence)

    # Check if the sequence is empty
    if num_jobs == 0:
        # Return a default value (you may choose 0 or another suitable value)
        return 0

    completion_times = np.zeros((num_jobs, num_machines))

    # Calculate the completion times for the first machine
    completion_times[0][0] = processing_times[sequence[0]][0]
    for i in range(1, num_jobs):
        completion_times[i][0] = completion_times[i-1][0] + processing_times[sequence[i]][0]

    # Calculate the completion times for the remaining machines
    for j in range(1, num_machines):
        completion_times[0][j] = completion_times[0][j-1] + processing_times[sequence[0]][j]
        for i in range(1, num_jobs):
            completion_times[i][j] = max(completion_times[i-1][j], completion_times[i][j-1]) + processing_times[sequence[i]][j]

    # Return the total completion time, which is the completion time of the last job in the last machine
    return completion_times[num_jobs-1][num_machines-1]


In [5]:
class TabuList:
    def __init__(self, max_size=20):
        self.max_size = max_size
        self.moves = []

    def add(self, move):
        if len(self.moves) >= self.max_size:
            self.moves.pop(0)
        self.moves.append(move)

    def is_tabu(self, move):
        return move in self.moves




In [18]:

import random

def generate_neighbor(sequence):
    neighbor_type = random.choice([1, 2,3])
    
    if neighbor_type == 1:#swap
        return swap_neighbor(sequence)
    elif neighbor_type == 2:#insert 1 individual job
        return insert_neighbor(sequence)
    elif neighbor_type == 3:#insert a block of length k
        return block_insert_neighbor(sequence)

def swap_neighbor(sequence):
    # Select random positions i and j
    i, j = random.sample(range(len(sequence)), 2)
    neighbor = sequence[:]
    neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
    return neighbor

def insert_neighbor(sequence):
    # Select random positions i and j
    i, j = random.sample(range(len(sequence)), 2)
    neighbor = sequence[:]
    job_to_insert = neighbor.pop(i)
    neighbor.insert(j, job_to_insert)
    return neighbor

def block_insert_neighbor(sequence):
    # Select random positions i, j, and k
    i = random.randint(0, len(sequence) - 1)
    j = random.randint(0, len(sequence))
    k = random.randint(1, len(sequence) - i)
    neighbor = sequence[:]
    block_to_insert = neighbor[i:i+k]
    neighbor = neighbor[:j] + block_to_insert + neighbor[j:]
    return neighbor

def generate_candidate_list(sequence):
    n = len(sequence)
    candidate_list_size = 2 * n
    candidate_list = [generate_neighbor(sequence) for _ in range(candidate_list_size)]
    return candidate_list

def select_best_neighbor(processing_times,candidate_list, current_solution, tabu_list,best_makespan):
    best_neighbor = None
    
    # Iterate through the candidate list
    for neighbor in candidate_list:
        # Calculate makespan for the neighbor
        neighbor_makespan = evaluate_sequence(neighbor, processing_times)     
        # Check if the neighbor is not tabu and improves the current solution
        if neighbor_makespan < best_makespan and not tabu_list.is_tabu(neighbor):
            best_neighbor = neighbor
            best_makespan = neighbor_makespan
    
    # If no improving move is found, examine the whole candidate list
    if best_neighbor is None:
        for neighbor in candidate_list:
            neighbor_makespan = evaluate_sequence(neighbor, processing_times)
            if neighbor_makespan < best_makespan:
                best_neighbor = neighbor
                best_makespan = neighbor_makespan
                
    return best_neighbor 



def recherche_tabou(processing_times,tabu_size, max_it, max_it_stagn):
    s0,best_cmax = neh_algorithm(processing_times)
    best_solution = s0
    tabu_list = TabuList(tabu_size)
    stagnation_count = 0  # Counter to track stagnation iterations

    for i in range(max_it):
        candidate_list = generate_candidate_list(s0)
        best_neighbor = select_best_neighbor(processing_times, candidate_list, s0, tabu_list, best_cmax,)

        # Check if there is no improving move
        if best_neighbor is None:
            stagnation_count += 1
            if stagnation_count >= max_it_stagn:
                break  # Terminate if stagnation persists
        else:
            stagnation_count = 0  # Reset stagnation counter
            # Update the current solution
            s0 = best_neighbor
            
            # Update the makespan of the best solution if a better solution is found
            current_makespan = evaluate_sequence(s0, processing_times)
            if current_makespan < best_cmax:
                best_solution = s0
                best_cmax = current_makespan
            # Update the tabu list
            tabu_list.add(s0)

    return best_solution, best_cmax , i

In [22]:
import numpy as np
import time
#-----------------------------------------------instance 2 pour 20*5
matrix = np.array([
    [26, 38, 27, 88, 95, 55, 54, 63, 23, 45, 86, 43, 43, 40, 37, 54, 35, 59, 43, 50],
    [59, 62, 44, 10, 23, 64, 47, 68, 54, 9, 30, 31, 92, 7, 14, 95, 76, 82, 91, 37],
    [78, 90, 64, 49, 47, 20, 61, 93, 36, 47, 70, 54, 87, 13, 40, 34, 55, 13, 11, 5],
    [88, 54, 47, 83, 84, 9, 30, 11, 92, 63, 62, 75, 48, 23, 85, 23, 4, 31, 13, 98],
    [69, 30, 61, 35, 53, 98, 94, 33, 77, 31, 54, 71, 78, 9, 79, 51, 76, 56, 80, 72]
])

processing_times=matrix.T
max_iterations = 100
max_iterations_stagnation = 10
tabu_size = 7
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times,tabu_size, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

Best Solution: [5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
Best Makespan: 1367.0
Elapsed time of 0.1320493221282959 seconds.
nb it  10


In [262]:
import numpy as np
#-----------------------------------------------instance 3 pour 20*5
matrix = np.array([
    [77, 94, 9, 57, 29, 79, 55, 73, 65, 86, 25, 39, 76, 24, 38, 5, 91, 29, 22, 27],
    [39, 31, 46, 18, 93, 58, 85, 58, 97, 10, 79, 93, 2, 87, 17, 18, 10, 50, 8, 26],
    [14, 21, 15, 10, 85, 46, 42, 18, 36, 2, 44, 89, 6, 3, 1, 43, 81, 57, 76, 59],
    [11, 2, 36, 30, 89, 10, 88, 22, 31, 9, 43, 91, 26, 3, 75, 99, 63, 83, 70, 84],
    [83, 13, 84, 46, 20, 33, 74, 42, 33, 71, 32, 48, 42, 99, 7, 54, 8, 73, 30, 75]
])

processing_times=matrix.T
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[2, 19, 17, 4, 8, 0, 16, 13, 15, 7, 6, 18, 9, 11, 10, 5, 3, 14, 12, 1]
1157.0
Best Solution: [2, 3, 15, 13, 19, 17, 18, 0, 6, 4, 9, 11, 10, 12, 5, 16, 8, 7, 14, 1]
Best Makespan: 1089.0
Elapsed time of 0.8567054271697998 seconds.
nb it  100


In [284]:
import numpy as np
#-----------------------------------------------instance 4 pour 20*5
matrix = np.array([
    [53, 19, 99, 62, 88, 93, 34, 72, 42, 65, 39, 79, 9, 26, 72, 29, 36, 48, 57, 95],
    [93, 79, 88, 77, 94, 39, 74, 46, 17, 30, 62, 77, 43, 98, 48, 14, 45, 25, 98, 30],
    [90, 92, 35, 13, 75, 55, 80, 67, 3, 93, 54, 67, 25, 77, 38, 98, 96, 20, 15, 36],
    [65, 97, 27, 25, 61, 24, 97, 61, 75, 92, 73, 21, 29, 3, 96, 51, 26, 44, 56, 31],
    [64, 38, 44, 46, 66, 31, 48, 27, 82, 51, 90, 63, 85, 36, 69, 67, 81, 18, 81, 72]
])

processing_times=matrix.T
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[12, 8, 15, 10, 14, 6, 9, 18, 16, 0, 4, 1, 11, 7, 13, 2, 19, 5, 3, 17]
1338.0
Best Solution: [12, 8, 15, 10, 14, 16, 6, 9, 18, 0, 1, 11, 4, 7, 5, 2, 19, 13, 3, 17]
Best Makespan: 1314.0
Elapsed time of 1.3902790546417236 seconds.
nb it  153


In [270]:
import numpy as np
#-----------------------------------------------instance 1 20*10
matrix = np.array([
    [74, 21, 58, 4, 21, 28, 58, 83, 31, 61, 94, 44, 97, 94, 66, 6, 37, 22, 99, 83],
    [28, 3, 27, 61, 34, 76, 64, 87, 54, 98, 76, 41, 70, 43, 42, 79, 88, 15, 49, 72],
    [89, 52, 56, 13, 7, 32, 32, 98, 46, 60, 23, 87, 7, 36, 26, 85, 7, 34, 36, 48],
    [60, 88, 26, 58, 76, 98, 29, 47, 79, 26, 19, 48, 95, 78, 77, 90, 24, 10, 85, 55],
    [54, 66, 12, 57, 70, 82, 99, 84, 16, 41, 23, 11, 68, 58, 30, 5, 5, 39, 58, 31],
    [92, 11, 54, 97, 57, 53, 65, 77, 51, 36, 53, 19, 54, 86, 40, 56, 79, 74, 24, 3],
    [9, 8, 88, 72, 27, 22, 50, 2, 49, 82, 93, 96, 43, 13, 60, 11, 37, 91, 84, 67],
    [4, 18, 25, 28, 95, 51, 84, 18, 6, 90, 69, 61, 57, 5, 75, 4, 38, 28, 4, 80],
    [25, 15, 91, 49, 56, 10, 62, 70, 76, 99, 58, 83, 84, 64, 74, 14, 18, 48, 96, 86],
    [15, 84, 8, 30, 95, 79, 9, 91, 76, 26, 42, 66, 70, 91, 67, 3, 98, 4, 71, 62]
])

processing_times=matrix.T
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[17, 4, 3, 13, 1, 19, 14, 11, 12, 18, 8, 9, 10, 16, 2, 6, 5, 7, 0, 15]
1675.0
Best Solution: [17, 4, 3, 12, 1, 11, 14, 8, 9, 16, 13, 2, 5, 18, 7, 19, 10, 6, 0, 15]
Best Makespan: 1598.0
Elapsed time of 1.7771189212799072 seconds.
nb it  100


In [271]:
import numpy as np
#-----------------------------------------------instance 2 20*10
matrix = np.array([
    [80, 13, 64, 77, 17, 78, 82, 4, 72, 93, 68, 25, 67, 80, 43, 93, 21, 33, 14, 30],
    [59, 83, 85, 85, 70, 35, 2, 76, 46, 72, 69, 46, 3, 57, 71, 77, 33, 49, 59, 82],
    [59, 70, 76, 10, 65, 19, 77, 86, 21, 75, 96, 3, 50, 57, 66, 84, 98, 55, 70, 32],
    [31, 64, 11, 9, 32, 58, 98, 95, 25, 4, 45, 60, 87, 31, 1, 96, 22, 95, 73, 77],
    [30, 88, 14, 22, 93, 48, 10, 7, 14, 91, 5, 43, 30, 79, 39, 34, 77, 81, 11, 10],
    [53, 19, 99, 62, 88, 93, 34, 72, 42, 65, 39, 79, 9, 26, 72, 29, 36, 48, 57, 95],
    [93, 79, 88, 77, 94, 39, 74, 46, 17, 30, 62, 77, 43, 98, 48, 14, 45, 25, 98, 30],
    [90, 92, 35, 13, 75, 55, 80, 67, 3, 93, 54, 67, 25, 77, 38, 98, 96, 20, 15, 36],
    [65, 97, 27, 25, 61, 24, 97, 61, 75, 92, 73, 21, 29, 3, 96, 51, 26, 44, 56, 31],
    [64, 38, 44, 46, 66, 31, 48, 27, 82, 51, 90, 63, 85, 36, 69, 67, 81, 18, 81, 72]
])
processing_times=matrix.T
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[11, 19, 16, 9, 0, 1, 13, 17, 12, 8, 14, 15, 10, 18, 6, 3, 4, 7, 2, 5]
1797.0
Best Solution: [11, 6, 18, 16, 19, 1, 12, 13, 8, 14, 15, 9, 10, 3, 0, 4, 7, 2, 5, 17]
Best Makespan: 1716.0
Elapsed time of 1.7700021266937256 seconds.
nb it  100


In [277]:
import numpy as np
#-----------------------------------------------instance 1 20*20
matrix = np.array([
    [50, 90, 39, 34, 66, 81, 27, 48, 46, 68, 48, 92, 78, 84, 93, 39, 43, 1, 65, 87],
    [78, 56, 9, 43, 84, 73, 66, 38, 83, 57, 97, 52, 77, 13, 12, 2, 65, 93, 39, 1],
    [36, 43, 10, 19, 55, 48, 85, 70, 82, 39, 91, 82, 85, 17, 6, 54, 87, 85, 4, 72],
    [85, 88, 60, 98, 4, 99, 53, 21, 33, 53, 63, 18, 45, 29, 43, 41, 80, 4, 31, 19],
    [9, 92, 98, 44, 51, 8, 31, 15, 47, 31, 80, 83, 20, 84, 69, 49, 93, 39, 13, 88],
    [75, 64, 96, 95, 22, 41, 26, 33, 68, 9, 81, 28, 61, 69, 37, 57, 36, 80, 96, 74],
    [46, 94, 6, 19, 20, 51, 85, 92, 43, 75, 70, 70, 36, 31, 76, 63, 89, 46, 25, 88],
    [73, 3, 56, 73, 80, 82, 36, 98, 90, 46, 10, 46, 65, 83, 75, 47, 61, 28, 59, 22],
    [71, 49, 36, 87, 8, 25, 76, 73, 80, 6, 6, 33, 79, 10, 93, 65, 26, 73, 42, 18],
    [7, 40, 33, 64, 5, 25, 89, 95, 58, 83, 28, 35, 74, 5, 6, 9, 3, 2, 35, 41],
    [49, 49, 15, 18, 65, 55, 1, 79, 10, 37, 77, 80, 79, 84, 93, 21, 85, 64, 46, 35],
    [3, 53, 59, 7, 65, 58, 24, 55, 26, 40, 89, 94, 51, 74, 54, 86, 22, 83, 19, 44],
    [60, 88, 15, 26, 11, 16, 55, 59, 81, 53, 92, 23, 55, 79, 13, 89, 2, 17, 97, 41],
    [12, 47, 46, 17, 43, 16, 91, 94, 73, 89, 12, 58, 25, 24, 55, 1, 67, 3, 1, 71],
    [75, 19, 60, 87, 27, 48, 72, 88, 48, 59, 74, 86, 49, 94, 15, 95, 41, 94, 15, 71],
    [31, 61, 47, 32, 34, 69, 32, 1, 1, 80, 19, 57, 98, 37, 31, 51, 66, 38, 62, 72],
    [70, 78, 41, 9, 47, 94, 26, 65, 17, 42, 59, 80, 7, 75, 63, 96, 7, 10, 47, 38],
    [20, 78, 38, 26, 64, 62, 11, 38, 68, 37, 74, 9, 65, 16, 38, 85, 50, 62, 39, 97],
    [88, 30, 34, 33, 21, 7, 94, 10, 73, 85, 82, 62, 99, 67, 61, 10, 4, 70, 31, 49],
    [9, 41, 22, 34, 83, 55, 3, 8, 75, 30, 57, 65, 89, 60, 90, 84, 74, 17, 2, 19]
])
processing_times=matrix.T
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[15, 13, 19, 17, 9, 8, 14, 0, 5, 6, 11, 12, 7, 10, 4, 1, 18, 2, 16, 3]
2456.0
Best Solution: [15, 17, 13, 9, 7, 12, 8, 14, 0, 5, 6, 4, 19, 11, 10, 1, 3, 16, 2, 18]
Best Makespan: 2344.0
Elapsed time of 3.6103389263153076 seconds.
nb it  100


In [137]:
import numpy as np
#-----------------------------------------------instance 2 20*20
matrix = np.array([
    [26, 38, 27, 88, 95, 55, 54, 63, 23, 45, 86, 43, 43, 40, 37, 54, 35, 59, 43, 50],
    [59, 62, 44, 10, 23, 64, 47, 68, 54, 9, 30, 31, 92, 7, 14, 95, 76, 82, 91, 37],
    [78, 90, 64, 49, 47, 20, 61, 93, 36, 47, 70, 54, 87, 13, 40, 34, 55, 13, 11, 5],
    [88, 54, 47, 83, 84, 9, 30, 11, 92, 63, 62, 75, 48, 23, 85, 23, 4, 31, 13, 98],
    [69, 30, 61, 35, 53, 98, 94, 33, 77, 31, 54, 71, 78, 9, 79, 51, 76, 56, 80, 72]
])

processing_times=matrix.T
max_iterations = 100
max_iterations_stagnation = 10
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
1367.0
Best Solution: [5, 6, 17, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13]
Best Makespan: 1366.0
Elapsed time of 0.16459298133850098 seconds.
nb it  17


In [71]:
import numpy as np
#-----------------------------------------------instance 2
matrix = np.array([
    [26, 38, 27, 88, 95, 55, 54, 63, 23, 45, 86, 43, 43, 40, 37, 54, 35, 59, 43, 50],
    [59, 62, 44, 10, 23, 64, 47, 68, 54, 9, 30, 31, 92, 7, 14, 95, 76, 82, 91, 37],
    [78, 90, 64, 49, 47, 20, 61, 93, 36, 47, 70, 54, 87, 13, 40, 34, 55, 13, 11, 5],
    [88, 54, 47, 83, 84, 9, 30, 11, 92, 63, 62, 75, 48, 23, 85, 23, 4, 31, 13, 98],
    [69, 30, 61, 35, 53, 98, 94, 33, 77, 31, 54, 71, 78, 9, 79, 51, 76, 56, 80, 72]
])

processing_times=matrix.T
max_iterations = 100
max_iterations_stagnation = 10
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
1367.0
Best Solution: [5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
Best Makespan: 1367.0
Elapsed time of 0.10571002960205078 seconds.
nb it  10


In [72]:
import numpy as np
#-----------------------------------------------instance 2
matrix = np.array([
    [26, 38, 27, 88, 95, 55, 54, 63, 23, 45, 86, 43, 43, 40, 37, 54, 35, 59, 43, 50],
    [59, 62, 44, 10, 23, 64, 47, 68, 54, 9, 30, 31, 92, 7, 14, 95, 76, 82, 91, 37],
    [78, 90, 64, 49, 47, 20, 61, 93, 36, 47, 70, 54, 87, 13, 40, 34, 55, 13, 11, 5],
    [88, 54, 47, 83, 84, 9, 30, 11, 92, 63, 62, 75, 48, 23, 85, 23, 4, 31, 13, 98],
    [69, 30, 61, 35, 53, 98, 94, 33, 77, 31, 54, 71, 78, 9, 79, 51, 76, 56, 80, 72]
])

processing_times=matrix.T
max_iterations = 100
max_iterations_stagnation = 10
start_time = time.time()
best_solution, best_makespan, nb_it = recherche_tabou(processing_times, max_iterations, max_iterations_stagnation)
elapsed_time = time.time() - start_time
print("Best Solution:", best_solution)
print("Best Makespan:", best_makespan)
print(f'Elapsed time of {elapsed_time} seconds.')
print ("nb it ",nb_it+1 )

[5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
1367.0
Best Solution: [5, 6, 14, 11, 12, 10, 2, 15, 19, 3, 16, 0, 8, 1, 9, 7, 18, 4, 13, 17]
Best Makespan: 1367.0
Elapsed time of 0.11271142959594727 seconds.
nb it  10


#Remarques :
 - Pour des petites tailles de la liste tabu: 